# Homework 4 — Question 2: CartPole
Taylor Belcher · CSCE883 Machine Learning

Run **DQN**, **REINFORCE**, and **Advantage Actor–Critic (A2C)** from the [supplied repository](https://github.com/jordanlei/deep-reinforcement-learning-cartpole). Save episode/reward figures and compare performance after running the experiments. No training results are prefilled.

Run cells in order. First follow the environment setup below, then run the short smoke check. The three full training cells can be run separately. Default: 500 episodes per algorithm per seed, three seeds, CPU, no rendering. Training may take several minutes or longer. Keep the `cartpole_upstream` folder beside this notebook.

The local source snapshots are Jordan Lei's supplied implementations, not newly written replacements. The loader below extracts their classes without executing command-line parsing or obsolete imports.

## Automatic setup
Run the next cell. It creates `.venv` beside this notebook and installs CPU-only PyTorch and the required packages there, including Jupyter kernel support. It works with externally managed system Python and does not require `python3-venv`'s pip bootstrap.

When it finishes, choose **Select Kernel → Python (CSCE883 HW4)**, or select the printed interpreter under **Python Environments**. Then continue with the imports cell below. This one-time kernel selection makes the notebook run inside the environment it installed. Internet access is needed for installation.

The setup cell must print **CPU-ONLY SETUP v2: torch==2.15.0+cpu**. If it does not, close and reopen the notebook from disk before running setup.

In [ ]:
import os
import subprocess
import sys
import venv
from pathlib import Path

setup_root = next((p.resolve() for p in [Path.cwd(), Path.cwd() / "Homework/Homework4"]
                   if (p / "cartpole_upstream").is_dir()), None)
if setup_root is None:
    raise FileNotFoundError("Start Jupyter in Homework4 or the course repository root.")
setup_env = setup_root / ".venv"
setup_python = setup_env / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not setup_python.exists():
    # Skip ensurepip, which is often unavailable on system-managed Linux Python.
    venv.EnvBuilder(with_pip=False).create(setup_env)

packages = ["pip", "ipykernel", "gymnasium>=1.0,<2",
            "numpy>=1.26,<3", "matplotlib>=3.8,<4"]
if Path(sys.prefix).resolve() == setup_env.resolve():
    pip_command = [str(setup_python), "-m", "pip"]
else:
    pip_command = [sys.executable, "-m", "pip", "--python", str(setup_python)]
# Ignore configured extra package indexes; require an explicitly CPU-tagged build.
install_env = {key: value for key, value in os.environ.items()
               if not key.startswith("PIP_")}
install_env["PIP_CONFIG_FILE"] = os.devnull
print("CPU-ONLY SETUP v2: torch==2.15.0+cpu", flush=True)
print("Installing into", setup_env, flush=True)
try:
    subprocess.run([*pip_command, "install", "torch==2.15.0+cpu",
                    "--index-url", "https://download.pytorch.org/whl/cpu"],
                   check=True, env=install_env)
    subprocess.run([*pip_command, "install", *packages,
                    "--index-url", "https://pypi.org/simple"],
                   check=True, env=install_env)
except KeyboardInterrupt:
    print("Setup interrupted. Stop any remaining installer before retrying.")
    raise
subprocess.run([str(setup_python), "-c",
                "import torch; assert torch.version.cuda is None, 'Expected CPU-only PyTorch'; "
                "print('Verified CPU-only PyTorch:', torch.__version__)"], check=True)
subprocess.run([str(setup_python), "-m", "ipykernel", "install", "--user",
                "--name", "csce883-hw4", "--display-name", "Python (CSCE883 HW4)"], check=True)
print("Setup complete.")
print("Select Kernel → Python (CSCE883 HW4)")
print("Or choose this interpreter:", setup_python)


In [ ]:
import ast
import copy
import csv
import hashlib
import json
import math
import random
import time
from collections import namedtuple
from pathlib import Path
from types import SimpleNamespace

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.distributions import Categorical
from IPython.display import display, Markdown

# Works when Jupyter starts here or in the course repository root.
ROOT = next((p for p in [Path.cwd(), Path.cwd() / "Homework/Homework4"]
             if (p / "cartpole_upstream").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from Homework4 or the course repo root.")
DEVICE = torch.device("cpu")
torch.set_num_threads(1)  # Tiny networks often run faster with one CPU thread.
print("torch", torch.__version__, "gymnasium", gym.__version__, "device", DEVICE)


## Experiment settings and adaptations
The original scripts use an unwrapped `CartPole-v0` and impose their own 500-step cap. Here we use `CartPole-v1` with a **500-step finite episode** for all algorithms; reward is +1 per step. Both a fall and reaching the horizon end the episodic return (no bootstrapping beyond the chosen horizon). This matches the original capped-return experiment rather than treating the time limit as an infinite-horizon continuation.

Changes to the supplied implementations:
- Adapt Gymnasium's reset/step signatures through a small wrapper; seed Python, NumPy, PyTorch, and the environment.
- Give DQN a separate frozen target network and update it every 40 **episodes** (the original aliases learner/target and checks the last step index). Preserve its effective dropout-disabled behavior: the original `.target.eval()` also sets the aliased learner to evaluation mode. Retain its final ReLU and other hyperparameters.
- Guard DQN batches containing only terminal transitions; collect loss values as numbers so saved histories do not retain computation graphs; guard division by zero in loss logging.
- Correct the original off-by-one smoothing window in all scripts. Our plots recompute rolling means directly from raw episode rewards.
- Use in-memory scalar logging instead of TensorBoard and skip unused video/CLI code. Keep the original policy/value networks, return calculations, dropout rates, learning rates, and loss formulas. The actor–critic's original NumPy entropy term is detached, so it does not supply an entropy gradient; this is retained rather than silently changing the algorithm.

This is a corrected, notebook-compatible run of the supplied code, not an exact reproduction of its original bugs. The methods use equal episode budgets and seeds, but retain different original learning rates. Report this when comparing them.

References: [DQN source](https://github.com/jordanlei/deep-reinforcement-learning-cartpole/blob/master/dqn_cartpole.py), [REINFORCE source](https://github.com/jordanlei/deep-reinforcement-learning-cartpole/blob/master/reinforce_cartpole.py), [actor–critic source](https://github.com/jordanlei/deep-reinforcement-learning-cartpole/blob/master/actor_critic_cartpole.py), [Gymnasium migration guide](https://gymnasium.farama.org/introduction/migration_guide/).

In [ ]:
EPISODES = 500
SEEDS = [0, 1, 2]  # Set [0] for a quicker preliminary run.
HORIZON = 500
SMOOTH = 20
FINAL_WINDOW = 100
EVAL_EPISODES = 20
GAMMA = 0.99
LEARNING_RATES = {"DQN": 0.001, "REINFORCE": 0.01, "A2C": 0.01}
FILES = {"DQN": "dqn_cartpole.py", "REINFORCE": "reinforce_cartpole.py",
         "A2C": "actor_critic_cartpole.py"}
assert EPISODES >= 1 and SEEDS and len(set(SEEDS)) == len(SEEDS)
OUT = ROOT / "q2_results" / time.strftime("%Y%m%d-%H%M%S")
OUT.mkdir(parents=True, exist_ok=False)
config = dict(episodes=EPISODES, seeds=SEEDS, horizon=HORIZON, gamma=GAMMA,
              learning_rates=LEARNING_RATES, smoothing=SMOOTH,
              final_window=FINAL_WINDOW, eval_episodes=EVAL_EPISODES,
              torch=torch.__version__, gymnasium=gym.__version__, numpy=np.__version__,
              source_sha256={name: hashlib.sha256((ROOT / "cartpole_upstream" / file).read_bytes()).hexdigest()
                             for name, file in FILES.items()})
(OUT / "config.json").write_text(json.dumps(config, indent=2))
results = {}
print("Output folder:", OUT)


In [ ]:
class LegacyEpisodeEnv:
    """Expose the old API for the supplied finite-horizon training loops."""
    def __init__(self, seed):
        self.env = gym.make("CartPole-v1", max_episode_steps=HORIZON)
        self.seed = seed
        self.first_reset = True
        self.action_space = self.env.action_space
        self.observation_space = self.env.observation_space
        self.action_space.seed(seed)

    def reset(self):
        observation, _ = self.env.reset(seed=self.seed if self.first_reset else None)
        self.first_reset = False
        return observation

    def step(self, action):
        observation, reward, terminated, truncated, info = self.env.step(action)
        return observation, reward, terminated or truncated, info

    def close(self):
        self.env.close()


class ScalarLog:
    def __init__(self, *args):
        pass

    def add_scalar(self, *args):
        pass


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def replace_checked(source, old, new):
    if old not in source:
        raise ValueError(f"Expected source fragment missing: {old!r}")
    return source.replace(old, new)


def load_algorithm(name, env):
    source = (ROOT / "cartpole_upstream" / FILES[name]).read_text()
    source = replace_checked(source, "range(500)", "range(HORIZON)")
    source = replace_checked(source, "smoothed_reward[-1*smooth: -1]", "smoothed_reward[-smooth:]")
    # History entries must be detached numbers, not tensors with retained graphs.
    for expression in ["loss", "a_loss", "c_loss * 100"]:
        source = source.replace(f".append({expression})", f".append(float({expression}))")
    if name == "DQN":
        source = replace_checked(source, "self.target = dqn", "self.target = copy.deepcopy(dqn)")
        source = replace_checked(source, "self.target.eval()",
                                 "self.target.eval()\n    self.learner.eval()\n    self.target.requires_grad_(False)")
        source = replace_checked(source, "next_states = torch.cat([s for s in batch.next_state if s is not None])",
                                 "next_states = [s for s in batch.next_state if s is not None]")
        source = replace_checked(source, "next_state_values[has_next_state] = self.target(next_states).max(1)[0].detach()",
                                 "if next_states:\n      next_state_values[has_next_state] = self.target(torch.cat(next_states)).max(1)[0].detach()")
        source = replace_checked(source, "if i % self.target_update == 0:",
                                 "if (episode + 1) % self.target_update == 0:")
        source = replace_checked(source, "c_loss += loss", "c_loss += float(loss)")
        source = replace_checked(source, "c_loss/c_samples", "c_loss/max(c_samples, 1)")
    tree = ast.parse(source)
    # Only definitions needed by the algorithms; no upstream imports/top-level CLI.
    selected = [node for node in tree.body if isinstance(node, ast.ClassDef)
                or (isinstance(node, ast.Assign)
                    and any(isinstance(t, ast.Name) and t.id == "Transition" for t in node.targets))]
    namespace = dict(torch=torch, nn=nn, F=F, optim=optim, Variable=Variable,
                     Categorical=Categorical, np=np, random=random, math=math,
                     namedtuple=namedtuple, copy=copy, device=DEVICE, env=env,
                     SummaryWriter=ScalarLog, HORIZON=HORIZON)
    exec(compile(ast.Module(body=selected, type_ignores=[]), FILES[name], "exec"), namespace)
    return SimpleNamespace(**namespace)


In [ ]:
def make_runner(name, seed, smoke=False):
    seed_everything(seed)
    env = LegacyEpisodeEnv(seed)
    module = load_algorithm(name, env)
    lr = LEARNING_RATES[name]
    if name == "DQN":
        model = module.DQN(4, 2).to(DEVICE)
        runner = module.Runner(model, nn.MSELoss(), lr=lr, gamma=GAMMA,
                               batch_size=8 if smoke else 128,
                               target_update=2 if smoke else 40)
        assert runner.target is not runner.learner
        assert not any(p.requires_grad for p in runner.target.parameters())
    elif name == "REINFORCE":
        model = module.Policy(4, 2).to(DEVICE)
        runner = module.Runner(model, optim.Adam(model.parameters(), lr=lr), gamma=GAMMA)
    else:
        actor, critic = module.Actor(4, 2).to(DEVICE), module.Critic(4).to(DEVICE)
        model = module.ActorCritic(actor, critic).to(DEVICE)
        runner = module.Runner(actor, critic, optim.Adam(actor.parameters(), lr=lr),
                               optim.Adam(critic.parameters(), lr=lr), gamma=GAMMA)
    return env, runner, model


def evaluate(name, model, seed):
    # Separate environment and RNG seed; training remains unaffected.
    seed_everything(seed + 10000)
    model.eval()  # Disable dropout for evaluation.
    env = LegacyEpisodeEnv(seed + 10000)
    rewards = []
    try:
        with torch.no_grad():
            for _ in range(EVAL_EPISODES):
                state, total, done = env.reset(), 0, False
                while not done:
                    if name == "DQN":
                        x = torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                        action = model(x).argmax(dim=1).item()
                    else:
                        policy = model if name == "REINFORCE" else model.actor
                        action = Categorical(policy(state)).sample().item()
                    state, reward, done, _ = env.step(action)
                    total += reward
                rewards.append(total)
    finally:
        env.close()
    return rewards


def run_one(name, seed, episodes, smoke=False):
    env, runner, model = make_runner(name, seed, smoke)
    start = time.perf_counter()
    try:
        runner.train(episodes=episodes, smooth=SMOOTH)
        rewards = np.asarray(runner.plots["Reward"], dtype=float)
        assert len(rewards) == episodes and np.isfinite(rewards).all()
        assert ((rewards >= 1) & (rewards <= HORIZON)).all()
        assert all(torch.isfinite(p).all() for p in model.parameters())
        if smoke:
            return rewards
        record = dict(algorithm=name, seed=seed, rewards=rewards.tolist(),
                      seconds=time.perf_counter() - start,
                      evaluation_rewards=evaluate(name, model, seed))
        prefix = OUT / f"{name.lower()}_seed{seed}"
        prefix.with_suffix(".json").write_text(json.dumps(record, indent=2))
        torch.save(model.state_dict(), prefix.with_suffix(".pt"))
        return record
    finally:
        env.close()


# Short validation only: these rewards are not homework experiment results.
for name in FILES:
    check = run_one(name, seed=123, episodes=5, smoke=True)
    print(name, "smoke check passed:", check.tolist())


## Full training
Each cell below trains one method from scratch for every configured seed. Results and model weights are saved after each run. Re-running a method replaces that method's files in this output folder. Re-run the settings cell to start a fresh output folder. The full experiment keeps the same episode budget for all methods; the smoke check above is separate.

In [ ]:
results["DQN"] = []
for seed in SEEDS:
    print("DQN, seed", seed)
    results["DQN"].append(run_one("DQN", seed, EPISODES))


In [ ]:
results["REINFORCE"] = []
for seed in SEEDS:
    print("REINFORCE, seed", seed)
    results["REINFORCE"].append(run_one("REINFORCE", seed, EPISODES))


In [ ]:
results["A2C"] = []
for seed in SEEDS:
    print("A2C, seed", seed)
    results["A2C"].append(run_one("A2C", seed, EPISODES))


## Episode/reward figures
Thin lines show individual seed runs. Heavy lines show the mean across seeds of the trailing 20-episode mean. The shaded range is ±1 standard deviation across seeds, not a confidence interval. Raw rewards and individual algorithm figures are saved for the report.

In [ ]:
missing = set(FILES) - set(results)
if missing or any(len(results.get(name, [])) != len(SEEDS) for name in FILES):
    raise RuntimeError("Finish all three full-training cells before comparing.")


def rolling_mean(values, window):
    values = np.asarray(values, dtype=float)
    sums = np.concatenate(([0.0], np.cumsum(values)))
    end = np.arange(1, len(values) + 1)
    start = np.maximum(0, end - window)
    return (sums[end] - sums[start]) / (end - start)


fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
fig_compare, ax_compare = plt.subplots(figsize=(9, 5))
raw_rows = []
for ax, (name, runs) in zip(axes, results.items()):
    rewards = np.asarray([run["rewards"] for run in runs])
    smoothed = np.asarray([rolling_mean(row, SMOOTH) for row in rewards])
    x = np.arange(1, rewards.shape[1] + 1)
    for run, row in zip(runs, rewards):
        ax.plot(x, row, alpha=0.25, linewidth=0.7)
        raw_rows.extend([name, run["seed"], int(ep), float(r)] for ep, r in zip(x, row))
    mean, std = smoothed.mean(axis=0), smoothed.std(axis=0)
    ax.plot(x, mean, color="black", label=f"Mean ({SMOOTH}-episode rolling)")
    ax.fill_between(x, mean - std, mean + std, color="gray", alpha=0.2)
    ax.set(title=name, xlabel="Training episode", ylim=(0, HORIZON + 10))
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
    line, = ax_compare.plot(x, mean, label=name)
    ax_compare.fill_between(x, mean - std, mean + std, color=line.get_color(), alpha=0.15)
    single, single_ax = plt.subplots(figsize=(8, 4))
    for row in rewards:
        single_ax.plot(x, row, alpha=0.2, linewidth=0.7)
    single_ax.plot(x, mean, color="black", label=f"{SMOOTH}-episode rolling mean across seeds")
    single_ax.set(title=f"{name}: episode rewards", xlabel="Training episode", ylabel="Total reward")
    single_ax.legend()
    single.tight_layout()
    for ext in ["png", "pdf"]:
        single.savefig(OUT / f"{name.lower()}_rewards.{ext}", dpi=180)
    plt.close(single)
axes[0].set_ylabel("Total episode reward")
fig.tight_layout()
ax_compare.set(title="CartPole: learning curve comparison", xlabel="Training episode",
               ylabel=f"{SMOOTH}-episode rolling mean reward", ylim=(0, HORIZON + 10))
ax_compare.legend()
ax_compare.grid(alpha=0.2)
fig_compare.tight_layout()
for ext in ["png", "pdf"]:
    fig.savefig(OUT / f"all_reward_curves.{ext}", dpi=180)
    fig_compare.savefig(OUT / f"comparison.{ext}", dpi=180)
with (OUT / "episode_rewards.csv").open("w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["algorithm", "seed", "episode", "reward"])
    writer.writerows(raw_rows)
plt.show()


## Compare measured performance
Primary metric: average reward over the final 100 training episodes (or all episodes for a shorter run), averaged across seeds. Also report average reward over the entire training run (learning efficiency), independent post-training evaluation reward, and training time. Evaluation uses greedy DQN actions and sampled policy actions with dropout disabled; it therefore differs from exploratory training behavior. Standard deviations below describe variation of per-seed means.

A method is not universally best because it wins this small experiment. Inspect learning speed, stability, final performance, and the seed variation. The original learning rates are not an equal hyperparameter search.

In [ ]:
summary = []
for name, runs in results.items():
    final = [np.mean(run["rewards"][-FINAL_WINDOW:]) for run in runs]
    whole = [np.mean(run["rewards"]) for run in runs]
    evaluation = [np.mean(run["evaluation_rewards"]) for run in runs]
    summary.append(dict(algorithm=name, final_training_mean=float(np.mean(final)),
                        final_training_std=float(np.std(final)),
                        whole_training_mean=float(np.mean(whole)),
                        evaluation_mean=float(np.mean(evaluation)),
                        evaluation_std=float(np.std(evaluation)),
                        mean_training_seconds=float(np.mean([run["seconds"] for run in runs]))))
summary.sort(key=lambda row: row["final_training_mean"], reverse=True)
with (OUT / "summary.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
    writer.writeheader()
    writer.writerows(summary)
lines = ["| Algorithm | Final training reward (mean ± SD) | Whole-run mean | Evaluation (mean ± SD) | Training seconds |",
         "|---|---:|---:|---:|---:|"]
for row in summary:
    lines.append(f"| {row['algorithm']} | {row['final_training_mean']:.1f} ± {row['final_training_std']:.1f} | "
                 f"{row['whole_training_mean']:.1f} | {row['evaluation_mean']:.1f} ± {row['evaluation_std']:.1f} | "
                 f"{row['mean_training_seconds']:.1f} |")
display(Markdown("\n".join(lines)))
leaders = [row["algorithm"] for row in summary
           if np.isclose(row["final_training_mean"], summary[0]["final_training_mean"])]
print("Highest measured final-window mean:", ", ".join(leaders))
print("Use the curves and seed variation to assess whether that difference is convincing.")
print("Saved figures, raw rewards, summary, settings, and weights to", OUT)


## Notes for the written answer
After the full run, describe:
1. The environment, episode budget, seeds, and code adaptations.
2. The episode/reward curves for all three methods.
3. Which method performed best under the stated metric, with measured numbers.
4. Any tradeoff between learning speed, final reward, and stability; note whether evaluation agrees with training.

**Write your interpretation here after inspecting the outputs.** Include an AI-use disclosure appropriate to your course requirements if you use this notebook in your submission.